# Multimodal Emotion Recognition (CREMA-D)
Replaced notebook with your provided code content.

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchaudio
import torchaudio.transforms as AT
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

video_root = '/kaggle/input/datasets/alenken/multimodal-emotion-recognition-ravdess/crema-d'
audio_root = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'

emotion_map = {
    'ANG': 'angry',
    'DIS': 'disgust',
    'FEA': 'fear',
    'HAP': 'happy',
    'NEU': 'neutral',
    'SAD': 'sad'
}

audio_data = []
for file in os.listdir(audio_root):
    if file.endswith('.wav'):
        file_key = file.split('.')[0]
        audio_data.append({
            'audio_key': file_key,
            'audio_path': os.path.join(audio_root, file),
            'emotion': emotion_map.get(file_key.split('_')[2], 'Unknown')
        })

df_audio = pd.DataFrame(audio_data)

video_data = []
for root, dirs, files in os.walk(video_root):
    for file in files:
        if file.endswith('.mp4'):
            video_key = "_".join(file.split('_')[:-1])
            video_data.append({
                'audio_key': video_key,
                'video_path': os.path.join(root, file)
            })

df_video = pd.DataFrame(video_data)
df_multimodal = pd.merge(df_audio, df_video, on='audio_key', how='inner')

class CremaDataset(Dataset):
    def __init__(self, df, seq_len=8):
        self.df = df
        self.seq_len = seq_len
        self.emotions = sorted(df['emotion'].unique())
        self.label_map = {label: i for i, label in enumerate(self.emotions)}

        self.audio_transform = nn.Sequential(
            AT.MelSpectrogram(sample_rate=16000, n_mels=128),
            AT.AmplitudeToDB()
        )

        self.video_transform = T.Compose([
            T.ToPILImage(),
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def _get_video(self, path):
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs = np.linspace(0, total - 1, self.seq_len, dtype=int)
        frames = []
        for i in range(total):
            ret, frame = cap.read()
            if i in idxs and ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(self.video_transform(frame))
        cap.release()
        while len(frames) < self.seq_len:
            frames.append(torch.zeros(3, 224, 224))
        return torch.stack(frames)

    def _get_audio(self, path):
        wav, sr = torchaudio.load(path)
        if sr != 16000:
            wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        spec = self.audio_transform(wav)
        spec = torch.nn.functional.interpolate(spec.unsqueeze(0), size=(128, 128)).squeeze(0)
        return spec

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self._get_video(row['video_path']), self._get_audio(row['audio_path']), self.label_map[row['emotion']]

class MultimodalLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.audio_net = timm.create_model('resnet18', pretrained=True, in_chans=1, num_classes=0)
        self.video_net = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0)

        self.lstm = nn.LSTM(input_size=512 + 192, hidden_size=256, num_layers=2, batch_first=True, dropout=0.2)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, v, a):
        b, s, c, h, w = v.shape
        v_feat = self.video_net(v.view(-1, c, h, w)).view(b, s, -1)
        a_feat = self.audio_net(a).unsqueeze(1).repeat(1, s, 1)

        combined = torch.cat((v_feat, a_feat), dim=2)
        out, _ = self.lstm(combined)
        return self.classifier(out[:, -1, :])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_df, test_df = train_test_split(df_multimodal, test_size=0.2, stratify=df_multimodal['emotion'], random_state=42)

train_loader = DataLoader(CremaDataset(train_df), batch_size=4, shuffle=True)
test_loader = DataLoader(CremaDataset(test_df), batch_size=4)

model = MultimodalLSTM(num_classes=len(train_df['emotion'].unique())).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

history = {'loss': [], 'acc': []}

for epoch in range(15):
    model.train()
    total_loss, correct = 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/15")
    for v, a, l in pbar:
        v, a, l = v.to(device), a.to(device), l.to(device)
        optimizer.zero_grad()
        out = model(v, a)
        loss = criterion(out, l)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == l).sum().item()
        pbar.set_postfix(loss=loss.item(), acc=correct/((pbar.n+1)*train_loader.batch_size))
    history['loss'].append(total_loss/len(train_loader))
    history['acc'].append(correct/len(train_df))

model.eval()
preds, targets = [], []
with torch.no_grad():
    for v, a, l in tqdm(test_loader, desc='Evaluating'):
        out = model(v.to(device), a.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        targets.extend(l.numpy())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.plot(history['loss']); plt.title('Training Loss')
plt.subplot(1, 2, 2); plt.plot(history['acc']); plt.title('Training Accuracy')
plt.show()

emo_labels = sorted(df_multimodal['emotion'].unique())
print(classification_report(targets, preds, target_names=emo_labels))
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(targets, preds, display_labels=emo_labels, ax=ax, cmap='Blues')
plt.show()